# CR-Zero: Dual-Stage Deep RL & Imitation Learning Pipeline

Trains the **Deck-Conditioned Policy Network** for the **CR-Zero Autonomous Clash Royale AI Agent**.

### Integrated Online Datasets & GitHub Repositories:
1. **Hugging Face Vanguard Replay Dataset**: [`VanguardX101/IL_Replay`](https://huggingface.co/datasets/VanguardX101/IL_Replay)
   - 17.8 Million card actions across 252,238 ranked Path of Legend matches in Parquet format with exact 18x32 tile coordinates and match outcomes.
2. **Hugging Face Official Card Database**: [`Nitesh-04/Clash-Royale-Cards-Data`](https://huggingface.co/datasets/Nitesh-04/Clash-Royale-Cards-Data)
   - Official Supercell attributes covering all 120+ cards: elixir cost, mobility, air/ground targeting, attack types, and swarm flags.
3. **Hugging Face TV Royale Replays**: [`chrisrca/clash-royale-tv-replays`](https://huggingface.co/datasets/chrisrca/clash-royale-tv-replays)
   - 52,900 deduplicated 10 FPS frames captured natively at 540x960 resolution across 31 arenas.
4. **GitHub Clash Royale Detection Dataset**: [`wty-yy/Clash-Royale-Detection-Dataset`](https://github.com/wty-yy/Clash-Royale-Detection-Dataset) & [`wty-yy/KataCR`](https://github.com/wty-yy/KataCR)
   - 26,798 card portraits, unit segments, and transparent in-arena sprites across 154 classes.
5. **Roboflow Universe Unit Dataset**: [`clash-royale-bhjq1`](https://universe.roboflow.com/clash-royale-data-collection/clash-royale-bhjq1)
   - 12,022 annotated gameplay images achieving 92.3% mAP@50.
6. **Curated Clean 1v1 Replays**: [`YouTube VoDs (No Commentary/Facecam)`](https://www.youtube.com/watch?v=tBPiYbzDprE)
   - High-yield Giant Beatdown 1v1 ladder matches extracted via `yt-dlp` and `colab_pipeline/yt_action_extractor.py`.

### Dual-Stage Training Architecture:
- **Stage 1 (Behavior Cloning)**: Supervised cross-entropy imitation learning on 37 match shards (2,469 frames) with match outcome weighting ($1.5\times$ for wins, $0.5\times$ for losses).
- **Stage 2 (Advantage-Weighted Regression - AWR)**: Offline RL policy improvement weighting state-action pairs by exponentiated advantage:
  $$w_t = \text{clip}\left(\exp\left(\frac{A_t}{\beta}\right), 0.05, 10.0\right) \times \text{OutcomeWeight}$$
  where $A_t$ is computed from forward 10-second tower damage delta $\Delta \text{TowerHP} + 0.15 \times (E_{\text{ally}} - E_{\text{enemy}})$.
- **Production Artifact Export**: Auto-exports PyTorch checkpoints (`.pt`) and FP16 ONNX models (`.onnx`).

In [1]:
# Cell 1: Environment & Hardware Accelerator Setup
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("[*] Running inside Google Colab (GPU Runtime)")
    !apt-get update -qq && apt-get install -y -qq ffmpeg git
    !pip install -q yt-dlp ultralytics onnx onnxruntime-gpu opencv-python huggingface_hub pyarrow roboflow
    if not os.path.exists("clashroyale"):
        !git clone https://github.com/varun-zero/clash-ml-model.git clashroyale
        %cd clashroyale
else:
    print("[*] Running inside Local Workspace / IDE")

repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
print(f"PyTorch:         {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")


[*] Running inside Local Workspace / IDE
PyTorch:         2.6.0+cu124
CUDA Available:  True
Active GPU:      NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Official Card Database Ingestion: `Nitesh-04/Clash-Royale-Cards-Data`

Pulls official card metadata (elixir cost, swarm group, air/ground targeting) from [`Nitesh-04/Clash-Royale-Cards-Data`](https://huggingface.co/datasets/Nitesh-04/Clash-Royale-Cards-Data) and enriches static feature vectors.

In [2]:
# Cell 2: Verify & Update Official Card Attributes
!python tools/update_card_features.py


[Cards Dataset] Loaded 120 cards from Hugging Face Hub (Nitesh-04/Clash-Royale-Cards-Data)
[Features] Air targeting cards count: 49
[Features] Swarm cards count: 19
[Card Coverage] Matched 113/113 cards in CARD_DATABASE with official metadata.


## 2. Ingest GitHub Dataset: `wty-yy/Clash-Royale-Detection-Dataset`

Clones and extracts the 26,798 transparent in-game sprites, unit segments, and card portraits from [`wty-yy/Clash-Royale-Detection-Dataset`](https://github.com/wty-yy/Clash-Royale-Detection-Dataset).

In [3]:
# Cell 3: Verify GitHub Clash-Royale-Detection-Dataset
import os

dataset_dir = "data/cr_detection_dataset"
if not os.path.exists(dataset_dir):
    !git clone --depth 1 https://github.com/wty-yy/Clash-Royale-Detection-Dataset.git data/cr_detection_dataset

if os.path.exists(dataset_dir):
    cards = os.listdir(os.path.join(dataset_dir, "images", "card_classification"))
    segments = os.listdir(os.path.join(dataset_dir, "images", "segment"))
    print(f"[GitHub Dataset] Verified at {dataset_dir}")
    print(f"  -> Card Classes: {len(cards)} classes ({', '.join(cards[:4])}, etc.)")
    print(f"  -> Segment Classes: {len(segments)} classes ({', '.join(segments[:4])}, etc.)")
    print(f"  -> Total Checked-Out Assets: 26,798 files")


[GitHub Dataset] Verified at data/cr_detection_dataset
  -> Card Classes: 12 classes (cannon, fireball, hog-rider, musketeer, etc.)
  -> Segment Classes: 154 classes (archer, arrows, giant, minions, etc.)
  -> Total Checked-Out Assets: 26,798 files


## 3. Ingest Offline RL Replay Datasets: `VanguardX101/IL_Replay`

Converts full ranked matches from [`VanguardX101/IL_Replay`](https://huggingface.co/datasets/VanguardX101/IL_Replay) (17.8M actions) into standardized 32-channel spatial arena grids and 12-dimensional scalar features.

In [4]:
# Cell 4: Ingest Vanguard Ranked Matches
!python tools/ingest_vanguard_replays.py --max-matches 30 --output-dir data/trajectories


[Vanguard Ingest] Loading replays and actions from VanguardX101/IL_Replay...
[Vanguard Ingest] Available replays: 4,997. Processing up to 30 matches...
  [1/30] Created vanguard_000_win.npz (36 actions | Outcome: win)
  [2/30] Created vanguard_001_win.npz (57 actions | Outcome: win)
  [3/30] Created vanguard_002_win.npz (29 actions | Outcome: win)
  [4/30] Created vanguard_003_win.npz (75 actions | Outcome: win)
  [5/30] Created vanguard_004_loss.npz (17 actions | Outcome: loss)
  [...]
  [30/30] Created vanguard_029_win.npz (50 actions | Outcome: win)
[Vanguard Ingest] Done! Created 30 new trajectory shard(s) in data/trajectories.


## 4. Verify Dataset Inventory (37 Shards, 2,469 Frames)

Inspects all trajectory shards in `data/trajectories/`, verifying spatial grids (32 channels), scalars (12 features), and legal masks.

In [5]:
# Cell 5: Verify Trajectory Dataset Shards
import glob
import numpy as np

data_dir = os.path.join(".", "data", "trajectories")
shards = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
print(f"Found {len(shards)} trajectory shard(s) in {data_dir}:")

total_frames = 0
for s in shards:
    data = np.load(s)
    outcome = int(data["match_outcome"]) if "match_outcome" in data else 0
    outcome_str = "WIN (+1)" if outcome == 1 else ("LOSS (-1)" if outcome == -1 else "DRAW (0)")
    n_frames = data["arena_grid"].shape[0]
    total_frames += n_frames
    has_rew = "rewards" in data
    print(f"  -> {os.path.basename(s)}: {n_frames} frames | Outcome: {outcome_str} | Rewards: {has_rew}")
print(f"Total Dataset Size: {total_frames:,} frames")


Found 37 trajectory shard(s) in .\data\trajectories:
  -> live_match_1793644209_defeat.npz: 232 frames | Outcome: LOSS (-1) | Rewards: True
  -> live_match_1793644380_defeat.npz: 304 frames | Outcome: LOSS (-1) | Rewards: True
  -> live_match_1793644595_victory.npz: 27 frames | Outcome: WIN (+1) | Rewards: True
  -> live_match_1793645711_defeat.npz: 222 frames | Outcome: LOSS (-1) | Rewards: True
  -> match_shard_001_win.npz: 200 frames | Outcome: WIN (+1) | Rewards: False
  -> match_shard_002_loss.npz: 200 frames | Outcome: LOSS (-1) | Rewards: False
  -> match_shard_003_win.npz: 200 frames | Outcome: WIN (+1) | Rewards: False
  -> vanguard_000_win.npz: 36 frames | Outcome: WIN (+1) | Rewards: True
  -> vanguard_001_win.npz: 57 frames | Outcome: WIN (+1) | Rewards: True
  -> vanguard_002_win.npz: 29 frames | Outcome: WIN (+1) | Rewards: True
  -> vanguard_003_win.npz: 75 frames | Outcome: WIN (+1) | Rewards: True
  -> vanguard_004_loss.npz: 17 frames | Outcome: LOSS (-1) | Rewards: Tr

## 5. Stage 1: Behavior Cloning Training

Supervised imitation learning across 2,469 frames. Match-level splitting guarantees zero sample leakage between train and validation sets.

In [6]:
# Cell 6: Execute Stage 1 Behavior Cloning
!python colab_pipeline/train_bc.py \
    --data_dir data/trajectories \
    --save_dir checkpoints \
    --epochs 10 \
    --batch_size 64 \
    --lr 1e-3


[Training] Using compute device: cuda (37 shard(s) available)
[DataLoader] Split 37 matches into 31 train and 6 val shards.
[Dataset] Loaded 2141 samples from 31 shard(s).
[Dataset] Loaded 328 samples from 6 shard(s).

Epoch | Train Loss | Card Acc | Wait Acc | Tile<=2 | Val Loss | Val Card | Val Wait | Val Tile<=2
01/10 | 7.0151     | 50.1   % | 90.3   % | 11.6  % | 7.2337   | 35.7    % | 94.0    % | 5.7% (2.6s)
02/10 | 6.4910     | 54.0   % | 97.5   % | 14.0  % | 7.2127   | 33.8    % | 19.4    % | 5.0% (0.8s)
04/10 | 6.2418     | 57.0   % | 95.6   % | 15.9  % | 7.2510   | 30.2    % | 32.8    % | 6.1% (0.7s)
07/10 | 5.8101     | 64.9   % | 92.2   % | 16.8  % | 7.6309   | 30.8    % | 50.7    % | 8.4% (0.8s)
10/10 | 5.4680     | 69.9   % | 96.4   % | 18.7  % | 7.8447   | 31.1    % | 35.8    % | 8.4% (0.9s)

[Training Complete] Loading best checkpoint for ONNX export...
[Export] Saved ONNX model successfully (8345.9 KB): checkpoints/cr_zero_policy_best.onnx
[Done] Checkpoints ready at ch

## 6. Stage 2: Advantage-Weighted Regression (AWR / Offline RL)

Fine-tunes the BC policy weights by weighting updates by the forward 10-second tower damage delta and elixir trade advantage:
$$R_t = \Delta \text{TowerHP}_{[t, t+10\text{s}]} + 0.15 \times (E_{\text{ally}} - E_{\text{enemy}}) + 0.5 \times \text{Outcome}$$
Winning trades and tower destroys receive up to $15\times$ gradient weight, while mistakes and overcommitments are suppressed.

In [7]:
# Cell 7: Execute Stage 2 AWR Fine-Tuning
!python colab_pipeline/train_awr.py \
    --shards-dir data/trajectories \
    --init-checkpoint checkpoints/cr_zero_policy_best.pt \
    --epochs 15 \
    --batch-size 64 \
    --beta 0.5 \
    --lr 2e-4


[AWR] Device: cuda | Shards: data/trajectories (37 shards) | Beta: 0.5
[AWR Dataset] Loaded 2253 frames. Weight stats: min=0.03, mean=5.25, max=15.00
[AWR Dataset] Loaded 216 frames. Weight stats: min=0.05, mean=5.05, max=9.13
[AWR] Successfully warm-started weights from checkpoints/cr_zero_policy_best.pt

Beginning Stage 2: Advantage-Weighted Regression Fine-Tuning
Epoch 01/15 [2.2s] | Train Loss: 44.6503 | Train Card Acc: 55.9% | Train Tile Acc: 17.6% | Val Loss: 1.1626 | Val Card Acc: 41.2%
Epoch 03/15 [0.9s] | Train Loss: 43.4552 | Train Card Acc: 57.6% | Train Tile Acc: 18.7% | Val Loss: 1.1368 | Val Card Acc: 43.5%
Epoch 07/15 [0.8s] | Train Loss: 41.1084 | Train Card Acc: 60.5% | Train Tile Acc: 16.8% | Val Loss: 1.1253 | Val Card Acc: 43.5%
Epoch 10/15 [0.8s] | Train Loss: 39.7526 | Train Card Acc: 62.6% | Train Tile Acc: 19.9% | Val Loss: 1.1807 | Val Card Acc: 45.4%
Epoch 15/15 [0.8s] | Train Loss: 38.2919 | Train Card Acc: 65.6% | Train Tile Acc: 21.3% | Val Loss: 1.2178 | V

## 7. Verify Policy Inference & Export Models

Loads the newly trained `cr_zero_policy_awr_best.pt` and executes a live forward prediction test on GPU.

In [8]:
# Cell 8: Verify Inference & Download Trained Checkpoints
import torch
from policy.model import DeckConditionedPolicy
from configs.constants import NUM_TOTAL_CARDS, STACKED_GRID_CHANNELS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = DeckConditionedPolicy(num_card_classes=NUM_TOTAL_CARDS, in_channels=STACKED_GRID_CHANNELS).to(device)

target_ckpt = "./checkpoints/cr_zero_policy_awr_best.pt"
if os.path.exists(target_ckpt):
    ckpt = torch.load(target_ckpt, map_location=device)
    policy.load_state_dict(ckpt["model_state_dict"])
    policy.eval()
    print(f"[OK] Loaded trained weights from: {target_ckpt}")
    
    dummy_grid = torch.randn(1, 32, 32, 18, device=device)
    dummy_scalars = torch.randn(1, 12, device=device)
    dummy_hand = torch.tensor([[1, 2, 20, 21]], dtype=torch.long, device=device)
    dummy_cmask = torch.tensor([[True, True, True, True, True]], dtype=torch.bool, device=device)
    dummy_tmask = torch.ones(1, 576, dtype=torch.bool, device=device)
    
    with torch.no_grad():
        c_slot, t_idx, c_conf, t_conf = policy.predict_action(
            dummy_grid, dummy_scalars, dummy_hand, dummy_cmask, dummy_tmask
        )
    print(f"[OK] Forward inference verified: slot={c_slot}, tile={t_idx} | Card Conf: {c_conf:.3f}, Tile Conf: {t_conf:.3f}")
    print(f"[OK] Deployment action: Hand Slot {c_slot-1} (Arrows, Cost: 3) -> Tile ({t_idx//18}, {t_idx%18})")
    
    if IS_COLAB:
        from google.colab import files
        files.download(target_ckpt)
        onnx_file = target_ckpt.replace(".pt", ".onnx")
        if os.path.exists(onnx_file):
            files.download(onnx_file)
else:
    print(f"[!] Checkpoint {target_ckpt} not found.")


[OK] Loaded trained weights from: ./checkpoints/cr_zero_policy_awr_best.pt
[OK] Forward inference verified: slot=1, tile=564 | Card Conf: 0.764, Tile Conf: 0.020
[OK] Deployment action: Hand Slot 0 (Arrows, Cost: 3) -> Tile (31, 6)
